In [9]:
import os
try:
    import google.colab
    os.system('git clone https://github.com/ethanresek/luminal-classifiers 2>/dev/null; cd /content/luminal-classifiers && git pull')
except ImportError:
    pass

In [10]:
import numpy as np
import pandas as pd
import sys
import joblib

from sklearn.feature_selection import mutual_info_classif
from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

In [11]:
FEATURE_COUNTS = [5, 10, 20, 50]
RANDOM_SEED = 42

In [12]:
try:
    from google.colab import drive
    sys.path.append('/content/luminal-classifiers')
    file_name = '/content/luminal-classifiers/models/tuned_rf_20260411_165750.joblib'
    print('Working in Colab')
except ImportError:
    file_name = 'models/tuned_rf_20260411_165750.joblib'
    print('Working locally')
finally:
    data = joblib.load(file_name)
    model = data['model']
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

Working locally


In [13]:
importances = model.feature_importances_
feature_names = X_train.columns
all_top_k_idx = []

for k in FEATURE_COUNTS:
    top_k_idx = np.argsort(importances)[-k:]
    all_top_k_idx.append(top_k_idx)
    top_k_names = feature_names[top_k_idx]
    print(f"Top {k}: {list(top_k_names)}")

Top 5: ['e2f7', 'cdc25a', 'ccnb1', 'aurka', 'cdk1']
Top 10: ['rad51', 'e2f2', 'lamb3', 'fancd2', 'chek1', 'e2f7', 'cdc25a', 'ccnb1', 'aurka', 'cdk1']
Top 20: ['diras3', 'pik3r1', 'hes6', 'kit', 'runx1', 'lama2', 'brca1', 'adgra2', 'nr2f1', 'egfr', 'rad51', 'e2f2', 'lamb3', 'fancd2', 'chek1', 'e2f7', 'cdc25a', 'ccnb1', 'aurka', 'cdk1']
Top 50: ['sbno1', 'ran', 'ppp2r2a', 'plagl1', 'prkcq', 'spen', 'ttyh1', 'dab2', 'prkcz', 'map3k1', 'pdgfrb', 'chek2', 'rps6kb1', 'msh6', 'stat5b', 'aph1b', 'jag1', 'mapt', 'tgfb3', 'akr1c3', 'arid5b', 'tgfbr3', 'foxo3', 'rps6ka2', 'rad51c', 'abcb1', 'cdk2', 'ccne1', 'spry2', 'folr1', 'diras3', 'pik3r1', 'hes6', 'kit', 'runx1', 'lama2', 'brca1', 'adgra2', 'nr2f1', 'egfr', 'rad51', 'e2f2', 'lamb3', 'fancd2', 'chek1', 'e2f7', 'cdc25a', 'ccnb1', 'aurka', 'cdk1']


In [14]:
searches = []

for arr in all_top_k_idx:
    print("------------------------------------------------------")
    print(f"Searching for hyperparameters for {len(arr)} features")
    print("------------------------------------------------------")
    X_train_k = X_train.iloc[:, arr]
    X_test_k = X_test.iloc[:, arr]

    rf = RandomForestClassifier(class_weight='balanced')
    p_dist = {
        'n_estimators': [100, 200, 500, 1000],
        'max_depth': [5, 10, 20, 30, None],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 5, 10],
        'max_features': ['sqrt', 'log2', 0.1, 0.25, 0.5],
        'max_leaf_nodes': [None, 50, 100, 200, 500],
        'min_impurity_decrease': [0.0, 0.001, 0.005, 0.01, 0.05],
        'max_samples': [0.5, 0.7, 0.8, 0.9, None],
        'criterion': ['gini', 'entropy', 'log_loss']
    }

    rand_search = RandomizedSearchCV(rf, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
    rand_search.fit(X_train_k, y_train)

    y_pred = rand_search.predict(X_test_k)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    print("Accuracies for Randomized Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    best = rand_search.best_params_

    grid_params = {}
    for key, val in best.items():
        if isinstance(val, str) or val is None:
            grid_params[key] = [val]
        elif isinstance(val, int):
            step = max(1, round(0.1 * val))
            grid_params[key] = [max(1, val - step), val, val + step]
        elif isinstance(val, float):
            step = max(0.005, round(0.1 * val, 4))
            grid_params[key] = [max(0, round(val - step, 4)), val, val + step]

    post_rand_rf = rand_search.best_estimator_

    grid_search = GridSearchCV(post_rand_rf, param_grid=grid_params, scoring='f1', cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED))
    grid_search.fit(X_train_k, y_train)

    y_pred = grid_search.predict(X_test_k)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    print("Accuracies for Grid Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    searches.append(grid_search)


------------------------------------------------------
Searching for hyperparameters for 5 features
------------------------------------------------------
Accuracies for Randomized Search CV
F1-score: 0.84375
ROC-AUC: 0.8275352112676057
Balanced Accuracy: 0.8275352112676057
Accuracies for Grid Search CV
F1-score: 0.8541666666666666
ROC-AUC: 0.8395774647887323
Balanced Accuracy: 0.8395774647887324
------------------------------------------------------
Searching for hyperparameters for 10 features
------------------------------------------------------
Accuracies for Randomized Search CV
F1-score: 0.8775510204081632
ROC-AUC: 0.8595774647887323
Balanced Accuracy: 0.8595774647887324
Accuracies for Grid Search CV
F1-score: 0.883248730964467
ROC-AUC: 0.8645774647887324
Balanced Accuracy: 0.8645774647887323
------------------------------------------------------
Searching for hyperparameters for 20 features
------------------------------------------------------
Accuracies for Randomized Search 

In [15]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
joblib.dump({
    'searches': searches,
    'feature_indices': all_top_k_idx,
    'feature_counts': FEATURE_COUNTS
}, f"models/rf_grid_searches_{timestamp}.joblib")

['models/rf_grid_searches_20260412_095134.joblib']